In [2]:
import time
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, desc, sum, avg
# Initialize
spark = SparkSession.builder.appName("SparkCompleteNotes").getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/13 06:17:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
df = spark.range(0,1000000).withColumn("value", col("id")%1000)

In [4]:
# optimized and caching 

In [19]:
opti_df = df.filter(col("value") > 500).filter(col("id") < 5000000).select("id" , "value")

In [20]:
opti_df.explain()

== Physical Plan ==
*(1) Project [id#0L, (id#0L % 1000) AS value#2L]
+- *(1) Filter (((id#0L % 1000) > 500) AND (id#0L < 5000000))
   +- *(1) Range (0, 1000000, step=1, splits=2)




In [21]:
start_time = time. time()
count_uncached = optimized_df.count() # Action triggers DAG
end_time = time. time()
print(f"1. Optimized Execution | Count: {count_uncached} | Time: {round(end_time - start_time, 4)} seconds")

1. Optimized Execution | Count: 499000 | Time: 0.2301 seconds


In [22]:
opti_df.cache()
stTime = time.time()

count_cached = opti_df.count()

enTime = time.time()

print(
    f"2. First Cached Execution | Count: {count_cached} | Time: {round(enTime - stTime, 4)} seconds"
)

[Stage 9:=============================>                             (1 + 1) / 2]

2. First Cached Execution | Count: 499000 | Time: 1.226 seconds


In [27]:
stTime = time.time()

count_cached2 = opti_df.count()

enTime = time.time()

print(
    f"3. Second Cached Execution | Count: {count_cached2} | Time: {round(enTime - stTime, 4)} seconds"
)

3. Second Cached Execution | Count: 499000 | Time: 0.1681 seconds


In [32]:
from pyspark import StorageLevel

opti_df.persist(StorageLevel.MEMORY_ONLY)  

stTime = time.time()

count_cached3 = opti_df.count()

enTime = time.time()

print(
    f"4. Persisted Execution | Count: {count_cached3} | Time: {round(enTime - stTime, 4)} seconds"
)

4. Persisted Execution | Count: 499000 | Time: 0.1397 seconds


26/06/13 07:07:43 WARN CacheManager: Asked to cache already cached data.
